# Experiment 1: Pix2Pix Baseline (TensorFlow / Keras)

**Architecture:** U-Net Generator + PatchGAN Discriminator (70x70 receptive field)  
**Task:** Given an empty room image + reference furniture image, generate the furnished room  
**Input:** 6 channels (room RGB + furniture RGB)  
**Output:** 3 channels (furnished room RGB)  
**Loss:** LSGAN + 100 * L1  

The model must learn **both** where to place the furniture and how to render it realistically — no placement mask is provided.

Best-of-both-worlds implementation merging the official TF Pix2Pix tutorial patterns with our improvements:
- **From official:** loop-based generator, two-input discriminator, dual `GradientTape`, `tf.train.Checkpoint`
- **From ours:** InstanceNorm, LSGAN loss, LR linear decay, custom `tf.data` pipeline, comprehensive evaluation (SSIM/PSNR/FID)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow.keras import layers

print(f'TensorFlow version: {tf.__version__}')
print(f'GPUs available: {len(tf.config.list_physical_devices("GPU"))}')

In [ ]:
# ---- Mount Google Drive ----
from google.colab import drive
drive.mount('/content/drive')

# ---- Locate the dataset ----
# The shared folder "processed-256" must be added to your Drive first:
#   1. Open: https://drive.google.com/drive/folders/1ztrTRmGrfgW7SMLiiEhApNx3WPGthZ-V
#   2. Right-click the "processed-256" folder → Organise → Add shortcut → My Drive
# Then it appears under /content/drive/MyDrive/processed-256/

POSSIBLE_ROOTS = [
    '/content/drive/MyDrive/processed-256',
    '/content/drive/MyDrive/693-project/data/processed-256',
    '/content/drive/MyDrive/693-project/processed-256',
    '/content/drive/MyDrive/693 project/processed-256',
]

DATA_ROOT = None
for path in POSSIBLE_ROOTS:
    if os.path.isfile(os.path.join(path, 'metadata.csv')):
        DATA_ROOT = path
        break

if DATA_ROOT is None:
    print('Dataset not found at any expected path. Attempting gdown download...')
    !pip install -q gdown
    !gdown --folder 1ztrTRmGrfgW7SMLiiEhApNx3WPGthZ-V -O /content/data --remaining-ok
    DATA_ROOT = '/content/data/processed-256'

    expected = ['train/input', 'train/target', 'train/furniture',
                'val/input',   'val/target',   'val/furniture',
                'test/input',  'test/target',  'test/furniture']
    missing = [d for d in expected if not os.path.isdir(os.path.join(DATA_ROOT, d))]
    if missing:
        raise FileNotFoundError(
            f'gdown missed these folders: {missing}\n'
            f'Fix: add the shared folder to your Drive as a shortcut (see instructions above), '
            f'then re-run this cell.'
        )

CSV_PATH = os.path.join(DATA_ROOT, 'metadata.csv')
print(f'Dataset root: {DATA_ROOT}')
print(f'Metadata CSV: {CSV_PATH}  ({os.path.getsize(CSV_PATH)/1024:.0f} KB)')

for split in ['train', 'val', 'test']:
    n_files = len(os.listdir(os.path.join(DATA_ROOT, split, 'input')))
    print(f'  {split}/input: {n_files} files')

# ---- Checkpoint & sample dirs (persistent on your Drive) ----
CHECKPOINT_DIR = '/content/drive/MyDrive/693-project/checkpoints/pix2pix_tf'
SAMPLE_DIR     = '/content/drive/MyDrive/693-project/samples/pix2pix_tf'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SAMPLE_DIR, exist_ok=True)

In [ ]:
# ---- Copy data to Colab's local SSD for 10-20x faster I/O ----
import shutil, time

LOCAL_DATA = '/content/local_data'

if not os.path.isdir(LOCAL_DATA):
    print('Copying dataset from Drive to local SSD (one-time, ~1-3 min)...')
    t0 = time.time()
    shutil.copytree(DATA_ROOT, LOCAL_DATA)
    elapsed = time.time() - t0
    print(f'Done in {elapsed:.0f}s')
else:
    print(f'Local copy already exists at {LOCAL_DATA}')

DATA_ROOT = LOCAL_DATA
CSV_PATH = os.path.join(DATA_ROOT, 'metadata.csv')

for split in ['train', 'val', 'test']:
    n = len(os.listdir(os.path.join(DATA_ROOT, split, 'input')))
    print(f'  {split}/input: {n} files (local SSD)')

In [ ]:
# ---- Hyperparameters ----
IMG_SIZE = 256
BATCH_SIZE = 8
LR = 2e-4
BETA1 = 0.5
LAMBDA_L1 = 100
NUM_EPOCHS = 200
SAVE_EPOCH = 10
SAMPLE_EPOCH = 5

## Dataset

Uses `tf.data` for efficient parallel loading and prefetching.  
Each sample is a triplet: empty room, furniture reference image, and target (furnished room).  
The model receives `[room | furniture]` (6 channels) as input and must learn where and how to place the furniture.

In [ ]:
df = pd.read_csv(CSV_PATH)

def rebase_path(p):
    """Rebase paths from the original pipeline output to the download location."""
    parts = p.replace('\\', '/').split('/')
    for i, part in enumerate(parts):
        if part in ('train', 'val', 'test'):
            return os.path.join(DATA_ROOT, *parts[i:])
    return p

for col in ['input_path', 'target_path', 'furniture_path']:
    df[col] = df[col].apply(rebase_path)

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df = df[df['split'] == 'val'].reset_index(drop=True)
test_df = df[df['split'] == 'test'].reset_index(drop=True)
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')


def _load_and_preprocess(path):
    raw = tf.io.read_file(path)
    img = tf.image.decode_png(raw, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return tf.cast(img, tf.float32) / 127.5 - 1.0


def load_sample(input_path, target_path, furniture_path):
    """Load (condition, target, room, furniture).
    condition = [room(3ch) | furniture(3ch)] = 6 channels."""
    room = _load_and_preprocess(input_path)
    target = _load_and_preprocess(target_path)
    furniture = _load_and_preprocess(furniture_path)

    condition = tf.concat([room, furniture], axis=-1)  # 6 channels
    return condition, target, room, furniture


def make_dataset(split_df, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((
        split_df['input_path'].values,
        split_df['target_path'].values,
        split_df['furniture_path'].values,
    ))
    if shuffle:
        ds = ds.shuffle(len(split_df))
    ds = ds.map(load_sample, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)
test_ds = make_dataset(test_df, shuffle=False)

In [ ]:
# Visualize a batch
for condition, target, room, furniture in train_ds.take(1):
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    for i in range(min(4, room.shape[0])):
        axes[0, i].imshow((room[i].numpy() + 1.0) / 2.0)
        axes[0, i].set_title('Empty room')
        axes[0, i].axis('off')
        axes[1, i].imshow((furniture[i].numpy() + 1.0) / 2.0)
        axes[1, i].set_title('Furniture reference')
        axes[1, i].axis('off')
        axes[2, i].imshow((target[i].numpy() + 1.0) / 2.0)
        axes[2, i].set_title('Target (furnished)')
        axes[2, i].axis('off')
    plt.tight_layout()
    plt.show()

## Exploratory Data Analysis

Overview of the processed dataset: split sizes, unique rooms, detection confidence, bounding box geometry, and sample triplets.

In [ ]:
print('=' * 50)
print('DATASET OVERVIEW')
print('=' * 50)

print(f'\nTotal samples: {len(df)}')
print(f'\nSamples per split:')
split_counts = df['split'].value_counts()
for split in ['train', 'val', 'test']:
    if split in split_counts.index:
        print(f'  {split:6s}: {split_counts[split]:5d}  ({split_counts[split]/len(df)*100:.1f}%)')

print(f'\nUnique rooms per split:')
for split in ['train', 'val', 'test']:
    n_rooms = df[df['split'] == split]['uuid'].nunique()
    n_images = len(df[df['split'] == split])
    print(f'  {split:6s}: {n_rooms:4d} rooms  ({n_images/n_rooms:.1f} images/room avg)')

print(f'\nTotal unique rooms: {df["uuid"].nunique()}')
print(f'Angles observed: {sorted(df["angle"].unique())}')

print(f'\nDetection confidence:')
print(f'  mean: {df["confidence"].mean():.3f}')
print(f'  min:  {df["confidence"].min():.3f}')
print(f'  max:  {df["confidence"].max():.3f}')
print(f'  std:  {df["confidence"].std():.3f}')

print(f'\nMetadata columns: {list(df.columns)}')
df.head()

In [ ]:
df['bbox_w'] = df['bbox_x2'] - df['bbox_x1']
df['bbox_h'] = df['bbox_y2'] - df['bbox_y1']
df['bbox_area'] = df['bbox_w'] * df['bbox_h']
df['bbox_cx'] = (df['bbox_x1'] + df['bbox_x2']) / 2
df['bbox_cy'] = (df['bbox_y1'] + df['bbox_y2']) / 2

sample_img = Image.open(df['target_path'].iloc[0])
img_w, img_h = sample_img.size
df['bbox_area_pct'] = df['bbox_area'] / (img_w * img_h) * 100
print(f'Original image size: {img_w} x {img_h}')

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].hist(df['confidence'], bins=30, color='steelblue', edgecolor='black', alpha=0.8)
axes[0, 0].set_title('Detection Confidence')
axes[0, 0].set_xlabel('Confidence')
axes[0, 0].set_ylabel('Count')

axes[0, 1].hist(df['bbox_w'], bins=30, color='coral', edgecolor='black', alpha=0.8)
axes[0, 1].set_title('Bbox Width (px)')
axes[0, 1].set_xlabel('Width')

axes[0, 2].hist(df['bbox_h'], bins=30, color='coral', edgecolor='black', alpha=0.8)
axes[0, 2].set_title('Bbox Height (px)')
axes[0, 2].set_xlabel('Height')

axes[1, 0].hist(df['bbox_area_pct'], bins=30, color='mediumpurple', edgecolor='black', alpha=0.8)
axes[1, 0].set_title('Bbox Area (% of image)')
axes[1, 0].set_xlabel('Area %')

scatter = axes[1, 1].scatter(df['bbox_cx'], df['bbox_cy'], c=df['split'].map({'train': 0, 'val': 1, 'test': 2}),
                              cmap='Set1', alpha=0.3, s=10)
axes[1, 1].set_title('Bbox Centers (position in image)')
axes[1, 1].set_xlabel('Center X')
axes[1, 1].set_ylabel('Center Y')
axes[1, 1].invert_yaxis()
axes[1, 1].set_xlim(0, img_w)
axes[1, 1].set_ylim(img_h, 0)

split_counts = df['split'].value_counts()
axes[1, 2].bar(split_counts.index, split_counts.values, color=['#2196F3', '#4CAF50', '#FF9800'], edgecolor='black')
axes[1, 2].set_title('Samples per Split')
axes[1, 2].set_ylabel('Count')
for i, (split, count) in enumerate(split_counts.items()):
    axes[1, 2].text(i, count + 10, str(count), ha='center', fontweight='bold')

plt.suptitle('Dataset Statistics', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nBbox size stats:')
print(f'  Width  — mean: {df["bbox_w"].mean():.0f}px, median: {df["bbox_w"].median():.0f}px, '
      f'range: [{df["bbox_w"].min():.0f}, {df["bbox_w"].max():.0f}]')
print(f'  Height — mean: {df["bbox_h"].mean():.0f}px, median: {df["bbox_h"].median():.0f}px, '
      f'range: [{df["bbox_h"].min():.0f}, {df["bbox_h"].max():.0f}]')
print(f'  Area % — mean: {df["bbox_area_pct"].mean():.1f}%, median: {df["bbox_area_pct"].median():.1f}%, '
      f'range: [{df["bbox_area_pct"].min():.1f}%, {df["bbox_area_pct"].max():.1f}%]')

In [ ]:
n_samples = 6
sample_rows = df.sample(n_samples, random_state=42)

fig, axes = plt.subplots(n_samples, 3, figsize=(14, 3.5 * n_samples))
col_titles = ['Empty Room (input)', 'With Furniture (target)', 'Cropped Furniture']

for i, (_, row) in enumerate(sample_rows.iterrows()):
    inp_img = Image.open(row['input_path']).convert('RGB')
    tgt_img = Image.open(row['target_path']).convert('RGB')
    fur_img = Image.open(row['furniture_path']).convert('RGB')

    axes[i, 0].imshow(inp_img)
    axes[i, 1].imshow(tgt_img)
    axes[i, 2].imshow(fur_img)

    bbox = [row['bbox_x1'], row['bbox_y1'], row['bbox_x2'], row['bbox_y2']]
    from matplotlib.patches import Rectangle
    rect = Rectangle((bbox[0], bbox[1]), bbox[2]-bbox[0], bbox[3]-bbox[1],
                      linewidth=2, edgecolor='lime', facecolor='none')
    axes[i, 1].add_patch(rect)

    axes[i, 0].set_ylabel(f'{row["split"]}\nconf={row["confidence"]:.2f}', fontsize=9)

    for j in range(3):
        axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title(col_titles[j], fontsize=13, fontweight='bold')

plt.suptitle('Sample Triplets from Dataset', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Model Architecture

Built with the Keras Functional API for clean skip connections in the U-Net generator.

In [ ]:
class InstanceNormalization(layers.Layer):
    """Instance Normalization (per-sample, per-channel normalization)."""

    def __init__(self, epsilon=1e-5, **kwargs):
        super().__init__(**kwargs)
        self.epsilon = epsilon

    def build(self, input_shape):
        self.scale = self.add_weight(name='scale', shape=(input_shape[-1],),
                                     initializer='ones', trainable=True)
        self.offset = self.add_weight(name='offset', shape=(input_shape[-1],),
                                      initializer='zeros', trainable=True)

    def call(self, x):
        mean, var = tf.nn.moments(x, axes=[1, 2], keepdims=True)
        return self.scale * (x - mean) / tf.sqrt(var + self.epsilon) + self.offset


WEIGHT_INIT = tf.keras.initializers.RandomNormal(stddev=0.02)


def downsample(filters, size=4, apply_norm=True, dropout=0.0):
    block = tf.keras.Sequential()
    block.add(layers.Conv2D(filters, size, strides=2, padding='same',
                            kernel_initializer=WEIGHT_INIT, use_bias=False))
    if apply_norm:
        block.add(InstanceNormalization())
    block.add(layers.LeakyReLU(0.2))
    if dropout > 0:
        block.add(layers.Dropout(dropout))
    return block


def upsample(filters, size=4, dropout=0.0):
    block = tf.keras.Sequential()
    block.add(layers.Conv2DTranspose(filters, size, strides=2, padding='same',
                                     kernel_initializer=WEIGHT_INIT, use_bias=False))
    block.add(InstanceNormalization())
    block.add(layers.ReLU())
    if dropout > 0:
        block.add(layers.Dropout(dropout))
    return block

### U-Net Generator

In [ ]:
def build_generator():
    """U-Net generator using loop-based encoder/decoder (standard TF pattern)."""
    inputs = layers.Input(shape=[IMG_SIZE, IMG_SIZE, 6])

    down_stack = [
        downsample(64, apply_norm=False),   # 128x128
        downsample(128),                     # 64x64
        downsample(256),                     # 32x32
        downsample(512, dropout=0.5),        # 16x16
        downsample(512, dropout=0.5),        # 8x8
        downsample(512, dropout=0.5),        # 4x4
        downsample(512, dropout=0.5),        # 2x2
        downsample(512, apply_norm=False, dropout=0.5),  # 1x1
    ]

    up_stack = [
        upsample(512, dropout=0.5),  # 2x2
        upsample(512, dropout=0.5),  # 4x4
        upsample(512, dropout=0.5),  # 8x8
        upsample(512, dropout=0.5),  # 16x16
        upsample(256),               # 32x32
        upsample(128),               # 64x64
        upsample(64),                # 128x128
    ]

    last = layers.Conv2DTranspose(
        3, 4, strides=2, padding='same',
        kernel_initializer=WEIGHT_INIT, activation='tanh'
    )

    x = inputs
    skips = []
    for down in down_stack:
        x = down(x)
        skips.append(x)

    skips = list(reversed(skips[:-1]))

    for up, skip in zip(up_stack, skips):
        x = up(x)
        x = layers.Concatenate()([x, skip])

    outputs = last(x)  # 256x256x3
    return tf.keras.Model(inputs, outputs, name='generator')

### PatchGAN Discriminator (70x70 receptive field)

In [ ]:
def build_discriminator():
    """PatchGAN with two separate inputs (standard TF pattern)."""
    inp = layers.Input(shape=[IMG_SIZE, IMG_SIZE, 6], name='condition')
    tar = layers.Input(shape=[IMG_SIZE, IMG_SIZE, 3], name='target')

    x = layers.Concatenate()([inp, tar])  # 7 channels

    x = layers.Conv2D(64, 4, strides=2, padding='same',
                       kernel_initializer=WEIGHT_INIT)(x)
    x = layers.LeakyReLU(0.2)(x)          # 128x128

    x = layers.Conv2D(128, 4, strides=2, padding='same',
                       kernel_initializer=WEIGHT_INIT, use_bias=False)(x)
    x = InstanceNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)          # 64x64

    x = layers.Conv2D(256, 4, strides=2, padding='same',
                       kernel_initializer=WEIGHT_INIT, use_bias=False)(x)
    x = InstanceNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)          # 32x32

    x = layers.ZeroPadding2D()(x)         # 34x34
    x = layers.Conv2D(512, 4, strides=1,
                       kernel_initializer=WEIGHT_INIT, use_bias=False)(x)
    x = InstanceNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)          # 31x31

    x = layers.ZeroPadding2D()(x)         # 33x33
    x = layers.Conv2D(1, 4, strides=1,
                       kernel_initializer=WEIGHT_INIT)(x)  # 30x30

    return tf.keras.Model(inputs=[inp, tar], outputs=x, name='discriminator')

## Initialize Models

In [ ]:
generator = build_generator()
discriminator = build_discriminator()

opt_g = tf.keras.optimizers.Adam(learning_rate=LR, beta_1=BETA1)
opt_d = tf.keras.optimizers.Adam(learning_rate=LR, beta_1=BETA1)

checkpoint = tf.train.Checkpoint(
    generator=generator,
    discriminator=discriminator,
    generator_optimizer=opt_g,
    discriminator_optimizer=opt_d,
)
ckpt_manager = tf.train.CheckpointManager(checkpoint, CHECKPOINT_DIR, max_to_keep=5)

if ckpt_manager.latest_checkpoint:
    checkpoint.restore(ckpt_manager.latest_checkpoint)
    print(f'Restored from {ckpt_manager.latest_checkpoint}')

generator.summary()
print()
discriminator.summary()

## Training

In [ ]:
@tf.function
def train_step(condition, target):
    # Separate tapes (official pattern) -- more memory-efficient than persistent
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        fake = generator(condition, training=True)

        disc_real = discriminator([condition, target], training=True)
        disc_fake = discriminator([condition, fake], training=True)

        # LSGAN losses (ours -- more stable than BCE)
        d_loss_real = tf.reduce_mean(tf.square(disc_real - 1.0))
        d_loss_fake = tf.reduce_mean(tf.square(disc_fake))
        d_loss = 0.5 * (d_loss_real + d_loss_fake)

        g_loss_gan = tf.reduce_mean(tf.square(disc_fake - 1.0))
        g_loss_l1 = tf.reduce_mean(tf.abs(fake - target))
        g_loss = g_loss_gan + LAMBDA_L1 * g_loss_l1

    gen_grads = gen_tape.gradient(g_loss, generator.trainable_variables)
    disc_grads = disc_tape.gradient(d_loss, discriminator.trainable_variables)

    opt_g.apply_gradients(zip(gen_grads, generator.trainable_variables))
    opt_d.apply_gradients(zip(disc_grads, discriminator.trainable_variables))

    return g_loss, d_loss, g_loss_l1

In [ ]:
def save_samples(epoch, generator, dataset, sample_dir, n=4):
    for condition, target, room, furniture in dataset.take(1):
        fake = generator(condition[:n], training=False)

        room_vis = (room[:n].numpy() + 1.0) / 2.0
        furn_vis = (furniture[:n].numpy() + 1.0) / 2.0
        fake_vis = np.clip((fake[:n].numpy() + 1.0) / 2.0, 0, 1)
        target_vis = (target[:n].numpy() + 1.0) / 2.0

    fig, axes = plt.subplots(4, n, figsize=(4 * n, 16))
    for i in range(n):
        axes[0, i].imshow(room_vis[i])
        axes[0, i].set_title('Empty Room')
        axes[0, i].axis('off')
        axes[1, i].imshow(furn_vis[i])
        axes[1, i].set_title('Furniture Ref')
        axes[1, i].axis('off')
        axes[2, i].imshow(fake_vis[i])
        axes[2, i].set_title('Generated')
        axes[2, i].axis('off')
        axes[3, i].imshow(target_vis[i])
        axes[3, i].set_title('Ground Truth')
        axes[3, i].axis('off')

    plt.suptitle(f'Epoch {epoch}', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(sample_dir, f'epoch_{epoch:04d}.png'), dpi=100)
    plt.show()
    plt.close()


def update_lr(optimizer, epoch, initial_lr, num_epochs):
    """Linear decay over the last 50% of training."""
    decay_start = num_epochs // 2
    if epoch >= decay_start:
        new_lr = initial_lr * (1.0 - (epoch - decay_start) / (num_epochs - decay_start))
        optimizer.learning_rate.assign(max(new_lr, 1e-7))

In [ ]:
history = {'g_loss': [], 'd_loss': [], 'l1_loss': [], 'val_l1': []}
steps_per_epoch = len(train_df) // BATCH_SIZE

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_g, epoch_d, epoch_l1 = 0.0, 0.0, 0.0
    n_batches = 0

    pbar = tqdm(train_ds, total=steps_per_epoch, desc=f'Epoch {epoch}/{NUM_EPOCHS}')
    for condition, target, _, _ in pbar:
        g_loss, d_loss, l1_loss = train_step(condition, target)
        epoch_g += g_loss.numpy()
        epoch_d += d_loss.numpy()
        epoch_l1 += l1_loss.numpy()
        n_batches += 1
        pbar.set_postfix({'G': f'{g_loss.numpy():.4f}', 'D': f'{d_loss.numpy():.4f}'})

    history['g_loss'].append(epoch_g / n_batches)
    history['d_loss'].append(epoch_d / n_batches)
    history['l1_loss'].append(epoch_l1 / n_batches)

    # Validation L1
    val_l1 = 0.0
    n_val = 0
    for vc, vt, _, _ in val_ds:
        vf = generator(vc, training=False)
        val_l1 += tf.reduce_mean(tf.abs(vf - vt)).numpy()
        n_val += 1
    history['val_l1'].append(val_l1 / max(n_val, 1))

    update_lr(opt_g, epoch, LR, NUM_EPOCHS)
    update_lr(opt_d, epoch, LR, NUM_EPOCHS)

    print(f'Epoch {epoch} | G: {history["g_loss"][-1]:.4f} | '
          f'D: {history["d_loss"][-1]:.4f} | '
          f'L1: {history["l1_loss"][-1]:.4f} | '
          f'Val L1: {history["val_l1"][-1]:.4f}')

    if epoch % SAMPLE_EPOCH == 0:
        save_samples(epoch, generator, val_ds, SAMPLE_DIR)

    if epoch % SAVE_EPOCH == 0:
        ckpt_manager.save()
        print(f'  Checkpoint saved at epoch {epoch}')

# Save final checkpoint + exportable weights
ckpt_manager.save()
generator.save_weights(os.path.join(CHECKPOINT_DIR, 'gen_final.weights.h5'))
discriminator.save_weights(os.path.join(CHECKPOINT_DIR, 'disc_final.weights.h5'))
print('Training complete.')

## Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

axes[0].plot(history['g_loss'])
axes[0].set_title('Generator Loss')
axes[0].set_xlabel('Epoch')

axes[1].plot(history['d_loss'])
axes[1].set_title('Discriminator Loss')
axes[1].set_xlabel('Epoch')

axes[2].plot(history['l1_loss'])
axes[2].set_title('Train L1')
axes[2].set_xlabel('Epoch')

axes[3].plot(history['val_l1'])
axes[3].set_title('Val L1')
axes[3].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, 'loss_curves.png'), dpi=150)
plt.show()

## Evaluation

Metrics computed with TensorFlow built-ins (SSIM, PSNR) and saved images for external FID.

In [ ]:
all_ssim, all_psnr = [], []

fid_real_dir = os.path.join(SAMPLE_DIR, 'fid_real')
fid_fake_dir = os.path.join(SAMPLE_DIR, 'fid_fake')
os.makedirs(fid_real_dir, exist_ok=True)
os.makedirs(fid_fake_dir, exist_ok=True)

img_idx = 0
for condition, target, _, _ in tqdm(test_ds, desc='Evaluating'):
    fake = generator(condition, training=False)

    fake_01 = (fake + 1.0) / 2.0
    target_01 = (target + 1.0) / 2.0

    ssim_vals = tf.image.ssim(fake_01, target_01, max_val=1.0)
    psnr_vals = tf.image.psnr(fake_01, target_01, max_val=1.0)
    all_ssim.extend(ssim_vals.numpy().tolist())
    all_psnr.extend(psnr_vals.numpy().tolist())

    for j in range(fake_01.shape[0]):
        real_img = tf.cast(tf.clip_by_value(target_01[j] * 255.0, 0, 255), tf.uint8)
        fake_img = tf.cast(tf.clip_by_value(fake_01[j] * 255.0, 0, 255), tf.uint8)
        tf.io.write_file(
            os.path.join(fid_real_dir, f'{img_idx:05d}.png'),
            tf.image.encode_png(real_img)
        )
        tf.io.write_file(
            os.path.join(fid_fake_dir, f'{img_idx:05d}.png'),
            tf.image.encode_png(fake_img)
        )
        img_idx += 1

print('\n===== Pix2Pix (TF) Test Metrics =====')
print(f'  SSIM:     {np.mean(all_ssim):.4f}  (higher is better)')
print(f'  PSNR:     {np.mean(all_psnr):.2f} dB  (higher is better)')
print('======================================')

In [ ]:
# FID: install pytorch-fid and compute (works even in a TF environment)
!pip install pytorch-fid -q
!python -m pytorch_fid {fid_real_dir} {fid_fake_dir}

## Final Results Visualization

In [ ]:
for condition, target, room, furniture in test_ds.take(1):
    n_show = min(8, condition.shape[0])
    fake = generator(condition[:n_show], training=False)

    room_vis = (room[:n_show].numpy() + 1.0) / 2.0
    furn_vis = (furniture[:n_show].numpy() + 1.0) / 2.0
    fake_vis = np.clip((fake[:n_show].numpy() + 1.0) / 2.0, 0, 1)
    target_vis = (target[:n_show].numpy() + 1.0) / 2.0

fig, axes = plt.subplots(4, n_show, figsize=(4 * n_show, 16))
row_labels = ['Empty Room', 'Furniture Ref', 'Generated', 'Ground Truth']
for i in range(n_show):
    axes[0, i].imshow(room_vis[i])
    axes[0, i].axis('off')
    axes[1, i].imshow(furn_vis[i])
    axes[1, i].axis('off')
    axes[2, i].imshow(fake_vis[i])
    axes[2, i].axis('off')
    axes[3, i].imshow(target_vis[i])
    axes[3, i].axis('off')

for ax, label in zip(axes[:, 0], row_labels):
    ax.set_ylabel(label, fontsize=14, rotation=90, labelpad=20)

plt.suptitle('Pix2Pix (TF) -- Test Set Results', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, 'final_results.png'), dpi=150, bbox_inches='tight')
plt.show()

## Side-by-Side Comparison: Room + Furniture → Generated vs Ground Truth

4-column view for each test sample: what the model saw (empty room + furniture reference), what it should produce (ground truth), and what it generated.

In [ ]:
n_compare = 8

for condition, target, room, furniture in test_ds.take(1):
    n_show = min(n_compare, condition.shape[0])
    fake = generator(condition[:n_show], training=False)

    room_vis   = (room[:n_show].numpy() + 1.0) / 2.0
    furn_vis   = (furniture[:n_show].numpy() + 1.0) / 2.0
    target_vis = (target[:n_show].numpy() + 1.0) / 2.0
    fake_vis   = np.clip((fake[:n_show].numpy() + 1.0) / 2.0, 0, 1)

fig, axes = plt.subplots(n_show, 4, figsize=(20, 4.5 * n_show))
col_titles = ['Empty Room', 'Furniture Reference', 'Ground Truth', 'Generated']

for i in range(n_show):
    axes[i, 0].imshow(room_vis[i])
    axes[i, 1].imshow(furn_vis[i])
    axes[i, 2].imshow(target_vis[i])
    axes[i, 3].imshow(fake_vis[i])

    ssim_val = tf.image.ssim(
        tf.constant(fake_vis[i:i+1]),
        tf.constant(target_vis[i:i+1]),
        max_val=1.0
    ).numpy()[0]
    psnr_val = tf.image.psnr(
        tf.constant(fake_vis[i:i+1]),
        tf.constant(target_vis[i:i+1]),
        max_val=1.0
    ).numpy()[0]
    axes[i, 3].set_xlabel(f'SSIM: {ssim_val:.3f} | PSNR: {psnr_val:.1f} dB', fontsize=10)

    for j in range(4):
        axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title(col_titles[j], fontsize=14, fontweight='bold')

plt.suptitle('Pix2Pix (TF) — Room + Furniture → Generated vs Ground Truth',
             fontsize=18, fontweight='bold', y=1.005)
plt.tight_layout()
plt.savefig(os.path.join(SAMPLE_DIR, 'comparison_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {os.path.join(SAMPLE_DIR, "comparison_results.png")}')